# 07 — Geologic Hazards and Evidence Boundaries

Hazard evidence is separated into geology-supported, public-soils-supported, field-supported, and unknown. This notebook does not derive a reservation-wide soil-hazard map from absent Pine Ridge SSURGO coverage.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"src").is_dir())
sys.path.insert(0, str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
import pandas as pd
import geopandas as gpd
import yaml
from IPython.display import display
from src.constants import REPO_ROOT as ROOT, OUTPUTS_DIR
from src.loaders import load_tribal_boundaries
from src.sovereignty import print_data_acknowledgment, generate_citations
with open(ROOT/"config"/"config.yaml") as stream: CONFIG = yaml.safe_load(stream)
primary = load_tribal_boundaries(["Pine Ridge"])
pine_ridge = primary[primary["NAME"] == "Pine Ridge"]

In [ ]:
print_data_acknowledgment(["usda_ssurgo", "usgs_3d_model"])

In [ ]:
from src.soil_evidence import evidence_register
register = evidence_register()
display(register)

## Geology-supported evidence

The Spangler model can support statements about modeled formation-top elevation and relative subsurface position. It cannot by itself supply soil texture, shrink–swell measurements, erodibility, or site-specific geotechnical properties.

In [ ]:
depth_figure = ROOT/"outputs"/"figures"/"04_depth_to_pierre_shale.png"
cross_section = ROOT/"outputs"/"04_pine_ridge_cross_section.csv"
display(pd.DataFrame([
    ["Depth-to-Pierre diagnostic", depth_figure.exists(), "geology-supported", "DEM/model datum and resolution uncertainty applies"],
    ["Modeled horizon cross-section", cross_section.exists(), "geology-supported", "regional model; not site investigation"],
], columns=["artifact", "available", "evidence_level", "constraint"]))

## Public-soils coverage gate

Expansive-soil, hydrologic-group, farmland, and erodibility outputs require verified local soil polygons and attributes. The gate prevents adjacent surveys from becoming implicit Pine Ridge estimates.

In [ ]:
from src.loaders import load_ssurgo_mapunits
from src.soil_evidence import require_coverage, SoilCoverageError
mapunits = load_ssurgo_mapunits()
try:
    require_coverage(mapunits, pine_ridge, "Pine Ridge public SSURGO")
    public_soil_hazards_allowed = True
except SoilCoverageError as exc:
    public_soil_hazards_allowed = False
    print(exc)
if not public_soil_hazards_allowed:
    print("No reservation-wide SSURGO soil-hazard classification is produced.")

## Field-supported evidence

Authorized field observations can support sampled-site interpretations after governance and quality gates. Unsampled Pine Ridge soils remain `unknown`; absence of data is not evidence of low hazard or uniform conditions.

In [ ]:
hazard_summary = pd.DataFrame([
    ["Pierre Shale proximity", "geology-supported", "Screening only; verify datum and investigate site"],
    ["Expansive soil", "unknown without authorized soil measurements", "Do not map reservation-wide"],
    ["Soil erodibility", "unknown without authorized soil measurements", "Do not infer from adjacent counties"],
    ["Site geotechnical suitability", "site investigation required", "No regional dataset substitutes for design investigation"],
], columns=["question", "current_evidence", "permitted_interpretation"])
display(hazard_summary)